# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sanaullah-Turab/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm picking **Lane 2: Refresh / Content Opportunity Scoring**.

The reason is simple: this is the one lane where this repo already has proof it works. The starter pipeline ran the full workflow on this exact question (rank pages for review) and the trained model beat the rule baseline by a wide margin. Precision@50 went from 0.240 (rule baseline) to 0.740 (random forest), see `outputs/model_report.md`. In plain terms, if a reviewer only has time to check the top 50 pages, the old rule gets about 12 of them right, the trained ranking gets about 37 right. That is not a small gap, and it is evidence I can point to before writing a single new line of code.

I also want a lane with enough candidate volume to actually rank against a review team's real capacity, not a handful of edge cases. Section 3 below shows that check on the starter data.


In [ ]:
lane_choice = {
    "lane": "Lane 2: Refresh / Content Opportunity Scoring",
    "reason": "starter pipeline already shows random forest beats baseline rules on precision@50 (0.740 vs 0.240)",
    "source": "outputs/model_report.md",
}
lane_choice


{'lane': 'Lane 2: Refresh / Content Opportunity Scoring',
 'reason': 'starter pipeline already shows random forest beats baseline rules on precision@50 (0.740 vs 0.240)',
 'source': 'outputs/model_report.md'}

## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a client's content inventory, which pages should a content reviewer look at first this week?

**Unit of analysis:** one page (`content_id`), scored and ranked within a client.

**Decision this improves:** how a reviewer with limited hours spends those hours across a large backlog of pages. Right now that decision either happens ad hoc or from a fixed rule (`baseline_refresh_score`). The question is whether a ranked, evidence-backed queue picks better candidates than the rule does.

**Who acts, and what they do:** a content reviewer (or the client-side SEO owner) opens the top of the ranked queue, reads the reason code attached to each page (e.g. `stale_visible_page`, `declining_with_demand`, `page_one_decay_risk`), and takes an action: refresh, expand, protect, or monitor.

**Output:** a ranked list of pages per client, each with a score, a reason code, and a suggested action.

**Cost of a wrong call:**
- False positive (page ranked high, but nothing is actually wrong): wasted reviewer hours on a page that didn't need attention. Annoying, but cheap, since the reviewer notices quickly and moves on.
- False negative (a real declining page with real demand never surfaces): the page keeps losing impressions and clicks unnoticed, and that loss compounds the longer it's missed. This is the more expensive error, since review capacity is fixed and a missed page doesn't get a second chance until the next scoring pass.

Because false negatives cost more than false positives here, precision@K (does the top of the queue hold real candidates) matters most, but I'll also watch recall on the declining set so I'm not silently starving it.

**Why data or ML helps at all:** the signals involved (impressions, position, CTR, freshness, engagement, trend) are individually simple, but a page can be a candidate for several overlapping reasons at once, and the weight each signal deserves shifts by content type and position tier. A single if-statement rule (the baseline) already exists and only gets 12 of its top 50 right. That gap between the rule and the trained model is exactly the kind of messy, tangled pattern that plain rules undersell and a model can learn.


In [ ]:
framing = {
    "decision": "which pages a reviewer with limited hours checks first this week",
    "actor": "content reviewer / SEO owner",
    "action": "refresh, expand, protect, or monitor, guided by the reason code",
    "cost_false_positive": "wasted reviewer hours on a healthy page",
    "cost_false_negative": "a real declining page keeps losing traffic, unnoticed until next pass",
    "primary_metric": "precision@K (K = review capacity), recall as a secondary check",
}
framing


{'decision': 'which pages a reviewer with limited hours checks first this week',
 'actor': 'content reviewer / SEO owner',
 'action': 'refresh, expand, protect, or monitor, guided by the reason code',
 'cost_false_positive': 'wasted reviewer hours on a healthy page',
 'cost_false_negative': 'a real declining page keeps losing traffic, unnoticed until next pass',
 'primary_metric': 'precision@K (K = review capacity), recall as a secondary check'}

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("rows:", len(df), "| columns:", df.shape[1])

# Reason-code style checks, straight from the lane guide's baseline definitions
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
page_one_decay_risk = ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).sum()
stale_visible_page = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()

print(f"declining_with_demand: {declining_with_demand} rows ({declining_with_demand/len(df):.1%} of the dataset)")
print(f"page_one_decay_risk:   {page_one_decay_risk} rows ({page_one_decay_risk/len(df):.1%} of the dataset)")
print(f"stale_visible_page:    {stale_visible_page} rows ({stale_visible_page/len(df):.1%} of the dataset)")


rows: 30000 | columns: 44
declining_with_demand: 13152 rows (43.8% of the dataset)
page_one_decay_risk:   7076 rows (23.6% of the dataset)
stale_visible_page:    17 rows (0.1% of the dataset)


## 4. Careful words: what I can and can't claim

**What this work can say:**
- Observed: on this starter slice, a trained ranking model finds more true declining/opportunity pages in its top 50 than the existing rule baseline does (37 vs 12, via client-holdout validation).
- Directional: certain signals (visibility trend, position, freshness, CTR relative to position tier) are associated with pages that need review; the model's feature importances point at which signals carry the most weight.
- Decision-support: the output is a ranked queue meant to help a human reviewer spend limited time better. It is a starting point for review, not a verdict.

**What this work cannot say:**
- It cannot prove that refreshing a flagged page causes traffic to recover. That needs an experiment (e.g. before/after with a control group), not a ranking model.
- It cannot claim to have found a Google ranking factor. The data only contains observed search and engagement signals, not anything about how Google's algorithm actually works.
- It cannot claim these exact numbers (0.740 precision@50) hold on the full ~79M-row warehouse. That result was earned on a 30k-row anonymized starter slice with client holdout; scaling up means re-earning it, not assuming it carries over.
- It cannot treat a declining label defined from the current window as a guarantee of future decline. A stronger version of this lane, once I move to the warehouse, should use a future-window label (prior 90 days of features predicting the next 30 days) instead of a same-window proxy.


In [ ]:
claims = {
    "can_claim": [
        "observed: trained ranking beats rule baseline on precision@50, on this starter slice",
        "directional: certain signals associate with review-worthy pages",
        "decision-support: ranked queue for a human reviewer, not a verdict",
    ],
    "cannot_claim": [
        "causal: refresh -> recovery (needs an experiment)",
        "a Google ranking factor",
        "that starter-slice numbers hold on the full warehouse without re-validation",
        "that a same-window proxy label is as strong as a future-window label",
    ],
}
claims


{'can_claim': ['observed: trained ranking beats rule baseline on precision@50, on this starter slice',
  'directional: certain signals associate with review-worthy pages',
  'decision-support: ranked queue for a human reviewer, not a verdict'],
 'cannot_claim': ['causal: refresh -> recovery (needs an experiment)',
  'a Google ranking factor',
  'that starter-slice numbers hold on the full warehouse without re-validation',
  'that a same-window proxy label is as strong as a future-window label']}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.